In [1]:
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, ggplot2, dplyr, lubridate)
library(readr)
library(magrittr)
library(stringr)
library(tidyr)

Loading required package: pacman


Attaching package: ‘magrittr’


The following object is masked from ‘package:purrr’:

    set_names


The following object is masked from ‘package:tidyr’:

    extract




# Read v2010 data and compute uncomp_care 

In [2]:
# Load the data
final.hcris.v2010 <- read_csv('../data/output/HCRIS_Data_v2010.csv')
 
# Step 1: Attempt to convert fy_end and check for NAs after conversion
final.hcris.v2010 <- final.hcris.v2010 %>%
  mutate(
    uncomp_care = tot_uncomp_care_charges - tot_uncomp_care_partial_pmts + bad_debt,
    fy_end = mdy(fy_end),
    fy_start = mdy(fy_start),  # Corrected from fy_strat to fy_start
    date_processed = mdy(date_processed),
    date_created = mdy(date_created),
    tot_discounts = abs(tot_discounts),  # Fixed the function call
    hrrp_payment = abs(hrrp_payment)
  ) %>%
  mutate(fyear = year(fy_end)) %>%
  arrange(provider_number, fyear) %>%
  select(-year)  # Make sure to select the correct column if you want to drop 'year'
 
# Now, group by fyear and count
final.hcris.v2010 %>%
  group_by(fyear) %>%
  count()

Rows: 57802 Columns: 48
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (12): provider_number, fy_start, fy_end, date_processed, date_created, d...
dbl (35): report, status, year, beds, tot_charges, tot_discounts, net_pat_re...
lgl  (1): npi

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


fyear,n
<dbl>,<int>
2011,5860
2012,6231
2013,6159
2014,6168
2015,6154
2016,6188
2017,6176
2018,6158
2019,6134


# Clean Duplicate Reports 

In [6]:
# Step 1: Count reports by provider and fiscal year
final.hcris.v2010 <- final.hcris.v2010 %>%
  add_count(provider_number, fyear, name = "total_reports")

In [7]:
# Step 2: Summarize the data, filtering out rows with NA in fyear
summary.hcris <- final.hcris.v2010 %>%
  filter(!is.na(fyear)) %>%
  group_by(provider_number, fyear) %>%
  mutate(
    hrrp_payment = if_else(is.na(hrrp_payment), 0, hrrp_payment),
    hvbp_payment = if_else(is.na(hvbp_payment), 0, hvbp_payment)
  ) %>%
  summarize(
    beds = max(beds, na.rm = TRUE),
    tot_charges = sum(tot_charges, na.rm = TRUE),
    tot_discounts = sum(tot_discounts, na.rm = TRUE),
    tot_operating_exp = sum(tot_operating_exp, na.rm = TRUE),
    ip_charges = sum(ip_charges, na.rm = TRUE),
    icu_charges = sum(icu_charges, na.rm = TRUE),
    ancillary_charges = sum(ancillary_charges, na.rm = TRUE),
    tot_discharges = sum(tot_discharges, na.rm = TRUE),
    mcare_discharges = sum(mcare_discharges, na.rm = TRUE),
    mcaid_discharges = sum(mcaid_discharges, na.rm = TRUE),
    tot_mcare_payment = sum(tot_mcare_payment, na.rm = TRUE),
    secondary_mcare_payment = sum(secondary_mcare_payment, na.rm = TRUE),
    hvbp_payment = sum(hvbp_payment, na.rm = TRUE),
    hrrp_payment = sum(hrrp_payment, na.rm = TRUE),
    fy_start = min(fy_start, na.rm = TRUE),
    fy_end = max(fy_end, na.rm = TRUE),
    date_processed = max(date_processed, na.rm = TRUE),
    date_created = min(date_created, na.rm = TRUE),
    street = first(street),
    city = first(city),
    state = first(state),
    zip = first(zip),
    county = first(county),
    uncomp_care = sum(uncomp_care, na.rm = TRUE),
    tot_uncomp_care_charges = sum(tot_uncomp_care_charges, na.rm = TRUE),
    tot_uncomp_care_partial_pmts = sum(tot_uncomp_care_partial_pmts, na.rm = TRUE),
    bad_debt = sum(bad_debt, na.rm = TRUE),
    cost_to_charge = first(cost_to_charge)
  ) %>%
  filter(!is.na(beds)) %>%  # Filter out groups with all NA in beds
  mutate(source = "total for year", .groups = "drop")

Warning message:
“There were 656 warnings in `summarize()`.
The first warning was:
ℹ In argument: `beds = max(beds, na.rm = TRUE)`.
ℹ In group 1013: `provider_number = "014011"`, `fyear = 2012`.
Caused by warning in `max()`:
! no non-missing arguments to max; returning -Inf
ℹ Run `dplyr::last_dplyr_warnings()` to see the 655 remaining warnings.”
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by provider_number and fyear.
ℹ Output is grouped by provider_number.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(provider_number, fyear))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


In [8]:
# Check for NA values in relevant columns
na_summary <- final.hcris.v2010 %>%
  summarise(
    na_beds = sum(is.na(beds)),
    na_tot_charges = sum(is.na(tot_charges)),
    na_tot_discounts = sum(is.na(tot_discounts)),
    na_tot_operating_exp = sum(is.na(tot_operating_exp)),
    na_hrrp_payment = sum(is.na(hrrp_payment)),
    na_hvbp_payment = sum(is.na(hvbp_payment))
  )
 
print(na_summary)

# A tibble: 1 × 6
  na_beds na_tot_charges na_tot_discounts na_tot_operating_exp na_hrrp_payment
    <int>          <int>            <int>                <int>           <int>
1     665           2248             2989                  642           38025
# ℹ 1 more variable: na_hvbp_payment <int>


In [9]:
# Step 4: Calculate elapsed time for hospitals with multiple reports
duplicate.hcris <- final.hcris.v2010 %>%
  filter(total_reports > 1) %>%
  mutate(time_diff = fy_end - fy_start) %>%
  group_by(provider_number, fyear) %>%
  mutate(total_days = sum(time_diff, na.rm = TRUE))  # Ensure total_days is created

In [10]:
# Step 5: Aggregate if total_days sums to around 365 days
unique.hcris2 <- duplicate.hcris %>%
  filter(total_days < 370) %>%
  group_by(provider_number, fyear) %>%
  summarize(
    beds = max(beds, na.rm = TRUE),
    tot_charges = sum(tot_charges, na.rm = TRUE),
    tot_discounts = sum(tot_discounts, na.rm = TRUE),
    tot_operating_exp = sum(tot_operating_exp, na.rm = TRUE),
    ip_charges = sum(ip_charges, na.rm = TRUE),
    icu_charges = sum(icu_charges, na.rm = TRUE),
    ancillary_charges = sum(ancillary_charges, na.rm = TRUE),
    tot_discharges = sum(tot_discharges, na.rm = TRUE),
    mcare_discharges = sum(mcare_discharges, na.rm = TRUE),
    mcaid_discharges = sum(mcaid_discharges, na.rm = TRUE),
    tot_mcare_payment = sum(tot_mcare_payment, na.rm = TRUE),
    secondary_mcare_payment = sum(secondary_mcare_payment, na.rm = TRUE),
    hvbp_payment = sum(hvbp_payment, na.rm = TRUE),
    hrrp_payment = sum(hrrp_payment, na.rm = TRUE),
    fy_start = min(fy_start, na.rm = TRUE),
    fy_end = max(fy_end, na.rm = TRUE),
    date_processed = max(date_processed, na.rm = TRUE),
    date_created = min(date_created, na.rm = TRUE),
    street = first(street),
    city = first(city),
    state = first(state),
    zip = first(zip),
    county = first(county),
    uncomp_care = sum(uncomp_care, na.rm = TRUE),
    tot_uncomp_care_charges = sum(tot_uncomp_care_charges, na.rm = TRUE),
    tot_uncomp_care_partial_pmts = sum(tot_uncomp_care_partial_pmts, na.rm = TRUE),
    bad_debt = sum(bad_debt, na.rm = TRUE),
    cost_to_charge = first(cost_to_charge)
  ) %>%
  filter(!is.na(beds)) %>%  # Filter out groups with all NA in beds
  mutate(source = "total for year", .groups = "drop")

Warning message:
“There was 1 warning in `summarize()`.
ℹ In argument: `beds = max(beds, na.rm = TRUE)`.
ℹ In group 213: `provider_number = "453308"`, `fyear = 2015`.
Caused by warning in `max()`:
! no non-missing arguments to max; returning -Inf”
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by provider_number and fyear.
ℹ Output is grouped by provider_number.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(provider_number, fyear))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


In [11]:
# Step 10: Calculate weighted averages for the remaining hospitals
# Check for NA values first
na_check <- duplicate.hcris3 %>%
  summarise(across(everything(), ~ sum(is.na(.)), .names = "na_{col}"))
 
print(na_check)
 
# Print intermediate data for verification
print(duplicate.hcris3)
 
# Summarize
unique.hcris4 <- duplicate.hcris3 %>%
  group_by(provider_number, fyear) %>%
  summarize(
    beds = max(beds, na.rm = TRUE),
    tot_charges = sum(tot_charges, na.rm = TRUE),
    tot_discounts = sum(tot_discounts, na.rm = TRUE),
    tot_operating_exp = sum(tot_operating_exp, na.rm = TRUE),
    ip_charges = sum(ip_charges, na.rm = TRUE),
    icu_charges = sum(icu_charges, na.rm = TRUE),
    ancillary_charges = sum(ancillary_charges, na.rm = TRUE),
    tot_discharges = sum(tot_discharges, na.rm = TRUE),
    mcare_discharges = sum(mcare_discharges, na.rm = TRUE),
    mcaid_discharges = sum(mcaid_discharges, na.rm = TRUE),
    tot_mcare_payment = sum(tot_mcare_payment, na.rm = TRUE),
    secondary_mcare_payment = sum(secondary_mcare_payment, na.rm = TRUE),
    hvbp_payment = sum(hvbp_payment, na.rm = TRUE),
    hrrp_payment = sum(hrrp_payment, na.rm = TRUE),
    fy_start = min(fy_start, na.rm = TRUE),
    fy_end = max(fy_end, na.rm = TRUE),
    date_processed = max(date_processed, na.rm = TRUE),
    date_created = min(date_created, na.rm = TRUE),
    street = first(street),
    city = first(city),
    state = first(state),
    zip = first(zip),
    county = first(county),
    uncomp_care = sum(uncomp_care, na.rm = TRUE),
    tot_uncomp_care_charges = sum(tot_uncomp_care_charges, na.rm = TRUE),
    tot_uncomp_care_partial_pmts = sum(tot_uncomp_care_partial_pmts, na.rm = TRUE),
    bad_debt = sum(bad_debt, na.rm = TRUE),
    cost_to_charge = first(cost_to_charge)
  ) %>%
  mutate(source = "weighted average", .groups = "drop")
 
# Print the results for verification
print(unique.hcris4)

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by provider_number and fyear.
ℹ Output is grouped by provider_number.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(provider_number, fyear))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


# A tibble: 588 × 54
# Groups:   provider_number [573]
   provider_number fyear na_report na_npi na_fy_start na_fy_end
   <chr>           <dbl>     <int>  <int>       <int>     <int>
 1 010018           2012         0      2           0         0
 2 010019           2014         0      2           0         0
 3 010022           2018         0      2           0         0
 4 010032           2017         0      2           0         0
 5 010038           2017         0      2           0         0
 6 010046           2015         0      2           0         0
 7 010054           2011         0      2           0         0
 8 010055           2014         0      2           0         0
 9 010069           2017         0      2           0         0
10 010078           2017         0      2           0         0
# ℹ 578 more rows
# ℹ 48 more variables: na_date_processed <int>, na_date_created <int>,
#   na_status <int>, na_data_source <int>, na_beds <int>, na_tot_charges <int>,
#   na_t

Warning message:
“There were 4 warnings in `summarize()`.
The first warning was:
ℹ In argument: `beds = max(beds, na.rm = TRUE)`.
ℹ In group 45: `provider_number = "033301"`, `fyear = 2017`.
Caused by warning in `max()`:
! no non-missing arguments to max; returning -Inf
ℹ Run `dplyr::last_dplyr_warnings()` to see the 3 remaining warnings.”
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by provider_number and fyear.
ℹ Output is grouped by provider_number.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(provider_number, fyear))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


# A tibble: 588 × 32
# Groups:   provider_number [573]
   provider_number fyear  beds tot_charges tot_discounts tot_operating_exp
   <chr>           <dbl> <dbl>       <dbl>         <dbl>             <dbl>
 1 010018           2012    12   72203507.     52722318.         24897842.
 2 010019           2014   185  317909935.    259258801          69371516 
 3 010022           2018    45   77336180.     65007991.         14946171.
 4 010032           2017    34   18222585.     14071822.          7023164.
 5 010038           2017   125  344555341.    316067599.         29476635.
 6 010046           2015   256  760393609.    683837396.         75778142.
 7 010054           2011   109  240108319.    203712711.         41313441.
 8 010055           2014   235  993595674.    828984910.        173206568.
 9 010069           2017    30   57159865.     40365359.         16992728.
10 010078           2017   250          0             0         155866673 
# ℹ 578 more rows
# ℹ 26 more variables: ip_c

# Save Final Per-Year CSVs

In [18]:
# Combine unique datasets
final.hcris.data <- bind_rows(unique.hcris2, unique.hcris3, unique.hcris4)
 
# Rename fyear to year and arrange the data
final.hcris.data <- final.hcris.data %>%
  rename(year = fyear) %>%
  arrange(provider_number, year)
# Write CSV files for each unique year
for (y in sort(unique(final.hcris.data$year))) {
  final.hcris.data %>%
    filter(year == y) %>%
    write_csv(paste0('../data/output/data-', y, '.csv'))
}
# Print summary information
cat("Years written:", sort(unique(final.hcris.data$year)), "\n")
cat("Total rows:", nrow(final.hcris.data), "\n")

Years written: 2011 2012 2013 2014 2015 2016 2017 2018 2019 
Total rows: 827 
